<a href="https://colab.research.google.com/github/papoochu/student-ai-learning-analysis/blob/main/notebooks/bertopic_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q bertopic
%pip install -q contractions
%pip install -q -U sentence-transformers

print("BERTopic libraries installed successfully!")


BERTopic libraries installed successfully!


In [3]:
import pandas as pd
from huggingface_hub import login

# Prompt for Hugging Face token to access the gated dataset
print("Please enter your Hugging Face Token (with READ access) to load the gated dataset:")
login()

# Load the StudyChat dataset
df = pd.read_json(
    "hf://datasets/wmcnicho/StudyChat/data.jsonl",
    lines=True
)

# Keep nonempty student prompts
corpus = df.copy()
corpus["prompt"] = corpus["prompt"].fillna("").astype(str)
corpus = corpus[
    corpus["prompt"].str.strip() != ""
].copy()

# Apply the same long-prompt threshold as previous models
corpus["word_count"] = corpus["prompt"].str.split().str.len()
modeling_corpus = corpus[
    corpus["word_count"] <= 1000
].copy()

# Preserve original text; only trim surrounding whitespace
docs = modeling_corpus["prompt"].str.strip().tolist()

print("Original interactions:", len(df))
print("Nonempty prompts:", len(corpus))
print("BERTopic documents:", len(docs))
if docs:
    print("\nExample prompt:")
    print(docs[0][:500])

Please enter your Hugging Face Token (with READ access) to load the gated dataset:


Original interactions: 16851
Nonempty prompts: 16850
BERTopic documents: 16597

Example prompt:
cross validation tests it on multiple sets, so does it randomly select data on my dataset? concise


In [4]:

# Set up sentence embeddings for BERTopic

import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)

print("Device:", device)
print("Embedding model:", "all-MiniLM-L6-v2")
print("Embedding dimensions:",
      embedding_model.get_sentence_embedding_dimension())
print("Documents ready:", len(docs))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Device: cpu
Embedding model: all-MiniLM-L6-v2
Embedding dimensions: 384
Documents ready: 16597


/tmp/ipykernel_889/4068280999.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension())


In [5]:

#Generate sentence embeddings

import numpy as np

embeddings = embedding_model.encode(
    docs,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding generation complete!")
print("Embeddings shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

# Save embeddings so we don't need to regenerate them
np.save("studychat_embeddings.npy", embeddings)

print("Embeddings saved to studychat_embeddings.npy")


Batches:   0%|          | 0/260 [00:00<?, ?it/s]

Embedding generation complete!
Embeddings shape: (16597, 384)
Data type: float32
Embeddings saved to studychat_embeddings.npy


In [6]:

#Train the default BERTopic model

from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=embedding_model,
    verbose=True
)

topics, probabilities = topic_model.fit_transform(
    docs,
    embeddings=embeddings
)

print("\nBERTopic training complete!")
print("Documents modeled:", len(topics))
print("Number of discovered topics:",
      len(set(topics)) - (1 if -1 in topics else 0))
print("Outlier documents:", topics.count(-1))

# Display the largest topics and their keywords
display(topic_model.get_topic_info().head(15))


2026-11-12 18:53:42,678 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-12 18:55:07,129 - BERTopic - Dimensionality - Completed ✓
2026-11-12 18:55:07,134 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-12 18:55:15,115 - BERTopic - Cluster - Completed ✓
2026-11-12 18:55:15,131 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-12 18:55:16,470 - BERTopic - Representation - Completed ✓



BERTopic training complete!
Documents modeled: 16597
Number of discovered topics: 460
Outlier documents: 3362


,Topic,Count,Name,Representation,Representative_Docs
0,-1,3362,-1_20_01_tree_false,"[20, 01, tree, false, node, word, prefix, true...",[here is the dataset: Chronic Kidney Disease\n...
1,0,155,0_git_commit_remote_github,"[git, commit, remote, github, changes, hint, p...",[yannguyen@vl965-172-31-187-129 assignment-4-e...
2,1,140,1_paragraph_sentences_answer_give,"[paragraph, sentences, answer, give, sentence,...","[give as paragraph, give me a paragraph, Answe..."
3,2,127,2_vocab_hyperparameters_stride_embeddings,"[vocab, hyperparameters, stride, embeddings, e...",[How should I choose these hyperparameters?\n#...
4,3,124,3_string_python_substring_strings,"[string, python, substring, strings, how, spac...","[how to get the length of a string in python, ..."
5,4,114,4_ai_film_jobs_people,"[ai, film, jobs, people, about, interesting, p...","[Write a short response (~250 words, max 500) ..."
6,5,109,5_code_do_me_way,"[code, do, me, way, exact, dont, give, can, wa...","[Can you give me code for this?, give me the c..."
7,6,99,6_poly_model_poly_polynomial_polynomialfeatures,"[poly, model_poly, polynomial, polynomialfeatu...",[# Using the PolynomialFeatures library perfor...
8,7,94,7_dictionary_check_python_key,"[dictionary, check, python, key, dictionaries,...","[what is a dictionary in python, what is a dic..."
9,8,89,8_word_node_build_tree_children,"[word, node, build_tree, children, char, docum...","[def build_tree(self, document):\n for ..."


In [7]:

#Summarize default BERTopic results

import pandas as pd

topic_info = topic_model.get_topic_info()

n_topics = (topic_info["Topic"] != -1).sum()
n_outliers = (pd.Series(topics) == -1).sum()
outlier_pct = 100 * n_outliers / len(docs)

print("Total documents:", len(docs))
print("Discovered topics (excluding -1):", n_topics)
print("Outlier documents:", n_outliers)
print(f"Outlier percentage: {outlier_pct:.2f}%")

print("\nTopic-size summary:")
display(
    topic_info.loc[
        topic_info["Topic"] != -1, "Count"
    ].describe().to_frame("Documents per topic")
)

print("\n20 largest topics:")
display(topic_info.head(21)[["Topic", "Count", "Name"]])


Total documents: 16597
Discovered topics (excluding -1): 460
Outlier documents: 3362
Outlier percentage: 20.26%

Topic-size summary:


,Documents per topic
count,460.000000
mean,28.771739
std,19.972150
min,10.000000
25%,16.000000
50%,22.500000
75%,35.000000
max,155.000000



20 largest topics:


,Topic,Count,Name
0,-1,3362,-1_20_01_tree_false
1,0,155,0_git_commit_remote_github
2,1,140,1_paragraph_sentences_answer_give
3,2,127,2_vocab_hyperparameters_stride_embeddings
4,3,124,3_string_python_substring_strings
5,4,114,4_ai_film_jobs_people
6,5,109,5_code_do_me_way
7,6,99,6_poly_model_poly_polynomial_polynomialfeatures
8,7,94,7_dictionary_check_python_key
9,8,89,8_word_node_build_tree_children


In [8]:

#Test BERTopic with larger clusters

from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP

# Set up UMAP with a fixed random seed
umap_model_30 = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# Encourage larger topic clusters
hdbscan_model_30 = HDBSCAN(
    min_cluster_size=30,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

# Train alternative BERTopic model
topic_model_30 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model_30,
    hdbscan_model=hdbscan_model_30,
    verbose=True
)

topics_30, probabilities_30 = topic_model_30.fit_transform(
    docs,
    embeddings=embeddings
)

# Summarize results
info_30 = topic_model_30.get_topic_info()

print("\nBERTopic: min_cluster_size = 30")
print("Number of topics:", (info_30["Topic"] != -1).sum())
print("Outlier documents:", topics_30.count(-1))
print(
    "Outlier percentage:",
    round(100 * topics_30.count(-1) / len(docs), 2),
    "%"
)

display(info_30.head(15))


2026-11-12 19:00:01,348 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-12 19:00:24,923 - BERTopic - Dimensionality - Completed ✓
2026-11-12 19:00:24,931 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-12 19:00:33,973 - BERTopic - Cluster - Completed ✓
2026-11-12 19:00:33,984 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-12 19:00:35,195 - BERTopic - Representation - Completed ✓



BERTopic: min_cluster_size = 30
Number of topics: 197
Outlier documents: 3645
Outlier percentage: 21.96 %


,Topic,Count,Name,Representation,Representative_Docs
0,-1,3645,-1_td_th_tensor_the,"[td, th, tensor, the, you, 82737724, and, to, ...",[Could you come up with a test loop for this m...
1,0,261,0_false_td_true_no,"[false, td, true, no, 4999999999999999, 000000...",[Convert this data to a markdown table for me....
2,1,232,1_pandas_pip_dataframe_requirement,"[pandas, pip, dataframe, requirement, packages...",[Requirement already satisfied: pandas in c:\U...
3,2,188,2_network_neural_hidden_layer_sizes_mlpclassifier,"[network, neural, hidden_layer_sizes, mlpclass...",[# Define different configurations\nnn_configs...
4,3,187,3_columns_rbc_categorical_columns_rbc_abnormal,"[columns, rbc, categorical_columns, rbc_abnorm...",[## Part 3.5: Encoding Categorical data\n\nIn ...
5,4,185,4_blood_cell_pus_red,"[blood, cell, pus, red, true, false, hemoglobi...",[age\t\t-\tage\t\n\t\t\tbp\t\t-\tblood pressur...
6,5,163,5_paragraph_sentences_answer_give,"[paragraph, sentences, answer, give, shorter, ...","[give in a short paragraph, give me the answer..."
7,6,160,6_hyperparameters_vocab_stride_embeddings,"[hyperparameters, vocab, stride, embeddings, e...",[How should I adjust these hyperparameters to ...
8,7,147,7_dictionary_dict_python_key,"[dictionary, dict, python, key, how, check, di...","[what is a dictionary in python?, what is a di..."
9,8,146,8_node_word_children_char,"[node, word, children, char, build_tree, docum...","[def build_tree(self, document):\n for ..."


In [9]:

#Compare BERTopic configurations

import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": "Default BERTopic",
        "Min Cluster Size": 10,
        "Topics": len(set(topics) - {-1}),
        "Outliers": topics.count(-1),
        "Outlier %": 100 * topics.count(-1) / len(docs)
    },
    {
        "Model": "Larger Clusters",
        "Min Cluster Size": 30,
        "Topics": len(set(topics_30) - {-1}),
        "Outliers": topics_30.count(-1),
        "Outlier %": 100 * topics_30.count(-1) / len(docs)
    }
])

display(comparison.round(2))


,Model,Min Cluster Size,Topics,Outliers,Outlier %
0,Default BERTopic,10,460,3362,20.26
1,Larger Clusters,30,197,3645,21.96


In [11]:
# Install gensim for coherence evaluation
%pip install -q gensim

#Evaluate BERTopic coherence and topic diversity

from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import re
import pandas as pd

# Tokenize original documents for coherence evaluation
stopwords = set(ENGLISH_STOP_WORDS)

tokenized_docs = []

for doc in docs:
    tokens = re.findall(
        r"\b[a-z_][a-z0-9_]*\b",
        doc.lower()
    )
    tokens = [
        token for token in tokens
        if token not in stopwords and len(token) > 2
    ]
    tokenized_docs.append(tokens)

dictionary = Dictionary(tokenized_docs)

bow_corpus = [
    dictionary.doc2bow(tokens)
    for tokens in tokenized_docs
]

def evaluate_bertopic(model):
    topic_words = []

    for topic_id in model.get_topics():

        if topic_id == -1:
            continue

        words = model.get_topic(topic_id)

        # Keep top 10 single-word terms in dictionary
        top_words = [
            word
            for word, weight in words
            if " " not in word and word in dictionary.token2id
        ][:10]

        if len(top_words) >= 2:
            topic_words.append(top_words)

    coherence = CoherenceModel(
        topics=topic_words,
        corpus=bow_corpus,
        dictionary=dictionary,
        coherence="u_mass"
    ).get_coherence()

    all_words = [
        word
        for topic in topic_words
        for word in topic
    ]

    diversity = (
        len(set(all_words)) / len(all_words)
        if all_words else 0
    )

    return coherence, diversity, len(topic_words)

results = []

for name, model in [
    ("Default BERTopic", topic_model),
    ("Larger Clusters", topic_model_30)
]:
    coherence, diversity, evaluated_topics = evaluate_bertopic(model)

    results.append({
        "Model": name,
        "UMass Coherence": coherence,
        "Topic Diversity": diversity,
        "Evaluated Topics": evaluated_topics
    })

bertopic_eval_df = pd.DataFrame(results)

display(bertopic_eval_df.round(4))

,Model,UMass Coherence,Topic Diversity,Evaluated Topics
0,Default BERTopic,-5.1219,0.5491,457
1,Larger Clusters,-3.4211,0.6058,197


In [12]:

#Train BERTopic with min_cluster_size = 50

umap_model_50 = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model_50 = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

topic_model_50 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model_50,
    hdbscan_model=hdbscan_model_50,
    verbose=True
)

topics_50, probabilities_50 = topic_model_50.fit_transform(
    docs,
    embeddings=embeddings
)

# Count topics and outliers
n_topics_50 = len(set(topics_50) - {-1})
n_outliers_50 = topics_50.count(-1)

# Use the evaluation function from Step 11
coherence_50, diversity_50, evaluated_50 = evaluate_bertopic(
    topic_model_50
)

print("BERTopic: min_cluster_size = 50")
print("Discovered topics:", n_topics_50)
print("Outlier documents:", n_outliers_50)
print(f"Outlier rate: {100 * n_outliers_50 / len(docs):.2f}%")
print(f"UMass coherence: {coherence_50:.4f}")
print(f"Topic diversity: {diversity_50:.4f}")
print("Evaluated topics:", evaluated_50)

display(topic_model_50.get_topic_info().head(12))


2026-11-12 19:08:19,151 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-12 19:09:00,635 - BERTopic - Dimensionality - Completed ✓
2026-11-12 19:09:00,640 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-12 19:09:10,474 - BERTopic - Cluster - Completed ✓
2026-11-12 19:09:10,485 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-12 19:09:11,628 - BERTopic - Representation - Completed ✓


BERTopic: min_cluster_size = 50
Discovered topics: 119
Outlier documents: 3657
Outlier rate: 22.03%
UMass coherence: -2.7437
Topic diversity: 0.6555
Evaluated topics: 119


,Topic,Count,Name,Representation,Representative_Docs
0,-1,3657,-1_td_feature_yes_th,"[td, feature, yes, th, true, false, and, tenso...",[<div>\n<style scoped>\n .dataframe tbody t...
1,0,261,0_false_td_true_no,"[false, td, true, no, 4999999999999999, 000000...",[convert the following data into a mark down t...
2,1,257,1_mid_frac_probability_aa,"[mid, frac, probability, aa, your, table, odot...",[What would probability table 2 and 3 be? I ha...
3,2,243,2_paragraph_sentences_give_answer,"[paragraph, sentences, give, answer, shorter, ...","[give as a paragraph, give me the answer in pa..."
4,3,232,3_pandas_pip_dataframe_packages,"[pandas, pip, dataframe, packages, requirement...",[pip install pandas install pandas\nDefaulting...
5,4,232,4_05_sklearn_nm_features,"[05, sklearn, nm, features, split, kcl, mols, ...",[can you do part 5\nPart 1. Equation of a Slim...
6,5,215,5_sql_query_target_count,"[sql, query, target, count, your_table_name, g...",[convert this to a pandas query:\n**SQL Query*...
7,6,212,6_outliers_remove_numerical_columns_mean,"[outliers, remove, numerical_columns, mean, ou...",[## Part 3.6 : Remove Outliers from Numerical ...
8,7,206,7_00_start_text_redacted_temperature,"[00, start_text, redacted, temperature, rnn_co...",[Generating Text with the Trained Model\nIn th...
9,8,204,8_x_embed_hidden_size_h_t_minus_1_hidden,"[x_embed, hidden_size, h_t_minus_1, hidden, ou...",[Does this look correct for Char RNN class Cha...


In [13]:

# Step 15: Inspect representative prompts from BERTopic (size 50)

import numpy as np
import pandas as pd

topic_info_50 = topic_model_50.get_topic_info()

# Ten largest discovered topics, excluding outliers
top_topic_ids = (
    topic_info_50[topic_info_50["Topic"] != -1]
    .nlargest(10, "Count")["Topic"]
    .tolist()
)

rng = np.random.default_rng(42)
sampled_prompts = []

for topic_id in top_topic_ids:

    indices = np.where(
        np.array(topics_50) == topic_id
    )[0]

    selected = rng.choice(
        indices,
        size=min(5, len(indices)),
        replace=False
    )

    keywords = [
        word for word, weight
        in topic_model_50.get_topic(topic_id)[:7]
    ]

    for idx in selected:
        sampled_prompts.append({
            "Topic": topic_id,
            "Keywords": ", ".join(keywords),
            "Prompt": docs[idx][:300]
        })

bertopic_samples_df = pd.DataFrame(sampled_prompts)

display(bertopic_samples_df)


,Topic,Keywords,Prompt
0,0,"false, td, true, no, 4999999999999999, 000000,...","convert this data into markdown table: age,bp,..."
1,0,"false, td, true, no, 4999999999999999, 000000,...",can you convert this into a markdown table \n\...
2,0,"false, td, true, no, 4999999999999999, 000000,...","convert this to a markdown table: age,bp,bgr,b..."
3,0,"false, td, true, no, 4999999999999999, 000000,...",unique_id age bp bgr bu ...
4,0,"false, td, true, no, 4999999999999999, 000000,...",nn1 nn2 nn3 nn4 nn5 ...
5,1,"mid, frac, probability, aa, your, table, odot",P(⊙∣Context)\tProbability value\nP(a∣aa)\t \n4...
6,1,"mid, frac, probability, aa, your, table, odot",what is the [0][0] and [0][1] when u print the...
7,1,"mid, frac, probability, aa, your, table, odot","for a string of length n, how many probabiliti..."
8,1,"mid, frac, probability, aa, your, table, odot","how many sequence ""aa"" in aababcaccaaacbaabcaa"
9,1,"mid, frac, probability, aa, your, table, odot","is this correct? $$P(X_1=a, X_2=b, X_3=c) = P(..."


In [14]:

#Check topic diversity across students and conversations

topic_analysis = modeling_corpus.reset_index(drop=True).copy()
topic_analysis["bertopic_topic"] = topics_50
topic_analysis["normalized_prompt"] = (
    topic_analysis["prompt"].str.strip().str.lower()
)

top_ten = (
    topic_model_50.get_topic_info()
    .query("Topic != -1")
    .head(10)["Topic"]
    .tolist()
)

summary = (
    topic_analysis[
        topic_analysis["bertopic_topic"].isin(top_ten)
    ]
    .groupby("bertopic_topic")
    .agg(
        Documents=("prompt", "size"),
        Unique_Students=("userId", "nunique"),
        Unique_Conversations=("chatId", "nunique"),
        Unique_Prompts=("normalized_prompt", "nunique")
    )
)

summary["Exact_Duplicate_Pct"] = (
    100 * (1 - summary["Unique_Prompts"] / summary["Documents"])
)

display(summary.round(2))


,Documents,Unique_Students,Unique_Conversations,Unique_Prompts,Exact_Duplicate_Pct
bertopic_topic,,,,,
0,261,126,186,257,1.53
1,257,83,113,254,1.17
2,243,96,178,237,2.47
3,232,82,154,228,1.72
4,232,95,132,209,9.91
5,215,162,177,195,9.30
6,212,98,125,194,8.49
7,206,73,94,199,3.40
8,204,65,97,200,1.96


In [15]:

#Prepare student-level samples for topic saturation

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Work with the exact documents used for BERTopic
saturation_df = modeling_corpus.reset_index(drop=True).copy()

# Get student IDs and shuffle them reproducibly
student_ids = saturation_df["userId"].dropna().unique()
shuffled_students = rng.permutation(student_ids)

fractions = [0.25, 0.50, 0.75, 1.00]
saturation_samples = {}
summary = []

for fraction in fractions:
    n_students = max(
        1,
        int(np.ceil(len(shuffled_students) * fraction))
    )

    selected_students = shuffled_students[:n_students]

    indices = np.flatnonzero(
        saturation_df["userId"].isin(selected_students).to_numpy()
    )

    saturation_samples[fraction] = indices

    summary.append({
        "Student Fraction": f"{fraction:.0%}",
        "Students": n_students,
        "Documents": len(indices),
        "Corpus Coverage (%)": round(
            100 * len(indices) / len(saturation_df), 2
        )
    })

display(pd.DataFrame(summary))

print("Total distinct students:", len(student_ids))
print("Samples are nested: smaller subsets are included in larger ones.")


,Student Fraction,Students,Documents,Corpus Coverage (%)
0,25%,51,3605,21.72
1,50%,102,7454,44.91
2,75%,153,12151,73.21
3,100%,203,16597,100.00


Total distinct students: 203
Samples are nested: smaller subsets are included in larger ones.


In [16]:

#Verify student-level saturation sample sizes

sample_summary = pd.DataFrame([
    {
        "Student Fraction": f"{fraction:.0%}",
        "Students": saturation_df.iloc[indices]["userId"].nunique(),
        "Documents": len(indices),
        "Dataset Coverage (%)": round(
            100 * len(indices) / len(saturation_df), 2
        )
    }
    for fraction, indices in saturation_samples.items()
])

display(sample_summary)

# Check that our nested samples are correctly constructed
fractions = [0.25, 0.50, 0.75, 1.00]

for smaller, larger in zip(fractions[:-1], fractions[1:]):
    assert set(saturation_samples[smaller]).issubset(
        set(saturation_samples[larger])
    )

assert len(saturation_samples[1.00]) == len(docs)

print("All saturation sample checks passed!")


,Student Fraction,Students,Documents,Dataset Coverage (%)
0,25%,51,3605,21.72
1,50%,102,7454,44.91
2,75%,153,12151,73.21
3,100%,203,16597,100.00


All saturation sample checks passed!


This experiment investigates whether the topics identified by BERTopic remain consistent as the number of students included in the dataset increases. To accomplish this, the StudyChat dataset was divided into four nested samples containing 25%, 50%, 75%, and 100% of the 203 students. Sampling was performed at the student level rather than the individual prompt level to keep each student's interactions together.

For the first experiment, we selected 51 students (25%), representing 3,605 prompts (21.72% of the dataset). We are training BERTopic on these prompts using the previously selected configuration (min_cluster_size=50, min_samples=10) and the existing sentence embeddings generated with all-MiniLM-L6-v2. UMAP reduces the dimensionality of the embeddings, HDBSCAN groups semantically similar prompts into clusters, and BERTopic identifies representative keywords for each topic.

The purpose is to compare the number of discovered topics, outlier percentages, and topic content across progressively larger samples. By examining whether similar themes continue to appear as more students are included, we can assess topic saturation and stability. This will help determine whether the discovered topics reflect recurring patterns in students' educational AI interactions or are strongly influenced by the particular students included in the analysis.

In [17]:

#Train BERTopic on 25% of students

from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP

# Select the 25% student sample
indices_25 = saturation_samples[0.25]

docs_25 = [docs[i] for i in indices_25]
embeddings_25 = embeddings[indices_25]

# Use the same configuration as our leading model
umap_25 = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_25 = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

topic_model_25 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_25,
    hdbscan_model=hdbscan_25,
    verbose=True
)

topics_25, probs_25 = topic_model_25.fit_transform(
    docs_25,
    embeddings=embeddings_25
)

# Summarize results
n_topics_25 = len(set(topics_25) - {-1})
n_outliers_25 = topics_25.count(-1)

print("\n25% Student Sample Results")
print("Students:", 51)
print("Documents:", len(docs_25))
print("Discovered topics:", n_topics_25)
print(f"Outlier rate: {100 * n_outliers_25 / len(docs_25):.2f}%")

display(
    topic_model_25.get_topic_info()[
        ["Topic", "Count", "Name"]
    ].head(12)
)


2026-11-12 19:33:55,862 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-12 19:34:34,691 - BERTopic - Dimensionality - Completed ✓
2026-11-12 19:34:34,694 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-12 19:34:34,811 - BERTopic - Cluster - Completed ✓
2026-11-12 19:34:34,821 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-12 19:34:35,086 - BERTopic - Representation - Completed ✓



25% Student Sample Results
Students: 51
Documents: 3605
Discovered topics: 17
Outlier rate: 8.54%


,Topic,Count,Name
0,-1,308,-1_self_torch_the_to
1,0,691,0_the_score_model_and
2,1,501,1_you_node_the_to
3,2,332,2_node_prefix_char_children
4,3,297,3_loss_epoch_20_optimizer
5,4,289,4_rows_pandas_duplicate_how
6,5,182,5_tensor_shape_torch_print
7,6,174,6_td_false_true_no
8,7,122,7_temperature_the_00_redacted
9,8,114,8_self_nn_hidden_size_torch


In [18]:

#Summarize the 25% BERTopic model

info_25 = topic_model_25.get_topic_info()
assigned_25 = info_25[info_25["Topic"] != -1].copy()

n_topics_25 = len(assigned_25)
n_outliers_25 = topics_25.count(-1)

top_5_count = assigned_25.nlargest(5, "Count")["Count"].sum()
assigned_count = len(docs_25) - n_outliers_25

print("25% Student Sample Summary")
print("Documents:", len(docs_25))
print("Discovered topics:", n_topics_25)
print("Outlier documents:", n_outliers_25)
print(f"Outlier rate: {100 * n_outliers_25 / len(docs_25):.2f}%")
print(f"Largest topic share: {100 * assigned_25['Count'].max() / len(docs_25):.2f}%")
print(f"Top 5 share of assigned documents: {100 * top_5_count / assigned_count:.2f}%")

print("\nTopic size distribution:")
display(assigned_25["Count"].describe().to_frame("Count"))


25% Student Sample Summary
Documents: 3605
Discovered topics: 17
Outlier documents: 308
Outlier rate: 8.54%
Largest topic share: 19.17%
Top 5 share of assigned documents: 64.00%

Topic size distribution:


,Count
count,17.000000
mean,193.941176
std,178.377574
min,51.000000
25%,77.000000
50%,114.000000
75%,289.000000
max,691.000000


In [19]:

#Train BERTopic on the 50% student sample

indices_50pct = saturation_samples[0.50]

docs_50pct = [docs[i] for i in indices_50pct]
embeddings_50pct = embeddings[indices_50pct]

umap_50pct = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_50pct = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

topic_model_50pct = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_50pct,
    hdbscan_model=hdbscan_50pct,
    verbose=True
)

topics_50pct, probs_50pct = topic_model_50pct.fit_transform(
    docs_50pct,
    embeddings=embeddings_50pct
)

# Summarize the results
info_50pct = topic_model_50pct.get_topic_info()
assigned_50pct = info_50pct[info_50pct["Topic"] != -1]

n_topics_50pct = len(assigned_50pct)
n_outliers_50pct = topics_50pct.count(-1)

top_5_count = assigned_50pct.nlargest(5, "Count")["Count"].sum()
assigned_count = len(docs_50pct) - n_outliers_50pct

print("\n50% Student Sample Results")
print("Students:", 102)
print("Documents:", len(docs_50pct))
print("Discovered topics:", n_topics_50pct)
print(f"Outlier rate: {100 * n_outliers_50pct / len(docs_50pct):.2f}%")
print(f"Largest topic share: {100 * assigned_50pct['Count'].max() / len(docs_50pct):.2f}%")
print(f"Top 5 share of assigned documents: {100 * top_5_count / assigned_count:.2f}%")

display(info_50pct[["Topic", "Count", "Name"]].head(12))


2026-11-12 19:40:07,384 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-12 19:40:24,991 - BERTopic - Dimensionality - Completed ✓
2026-11-12 19:40:24,993 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-12 19:40:25,248 - BERTopic - Cluster - Completed ✓
2026-11-12 19:40:25,254 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-12 19:40:25,762 - BERTopic - Representation - Completed ✓



50% Student Sample Results
Students: 102
Documents: 7454
Discovered topics: 43
Outlier rate: 20.79%
Largest topic share: 6.32%
Top 5 share of assigned documents: 30.72%


,Topic,Count,Name
0,-1,1550,-1_the_to_torch_and
1,0,471,0_you_code_me_give
2,1,397,1_rows_pandas_python_packages
3,2,380,2_tensor_shape_01_print
4,3,318,3_python_string_node_if
5,4,248,4_network_neural_score_hidden_layer_sizes
6,5,197,5_outliers_numerical_numerical_columns_normalize
7,6,186,6_td_false_true_no
8,7,178,7_blood_yes_cell_feature
9,8,178,8_jupyter_git_how_commit


In [20]:

#Evaluate the 50% saturation model

info_50pct = topic_model_50pct.get_topic_info()
assigned_50pct = info_50pct[
    info_50pct["Topic"] != -1
]

n_topics_50pct = len(assigned_50pct)
n_outliers_50pct = topics_50pct.count(-1)

top_5_count = (
    assigned_50pct.nlargest(5, "Count")["Count"].sum()
)
assigned_count = len(docs_50pct) - n_outliers_50pct

# Reuse our existing coherence/diversity function
coherence_50pct, diversity_50pct, evaluated_50pct = (
    evaluate_bertopic(topic_model_50pct)
)

print("50% Student Sample Summary")
print("Documents:", len(docs_50pct))
print("Discovered topics:", n_topics_50pct)
print(f"Outlier rate: {100*n_outliers_50pct/len(docs_50pct):.2f}%")
print(f"Largest topic share: {100*assigned_50pct['Count'].max()/len(docs_50pct):.2f}%")
print(f"Top 5 share of assigned documents: {100*top_5_count/assigned_count:.2f}%")
print(f"UMass coherence: {coherence_50pct:.4f}")
print(f"Topic diversity: {diversity_50pct:.4f}")
print("Topics evaluated:", evaluated_50pct)


50% Student Sample Summary
Documents: 7454
Discovered topics: 43
Outlier rate: 20.79%
Largest topic share: 6.32%
Top 5 share of assigned documents: 30.72%
UMass coherence: -1.8469
Topic diversity: 0.7281
Topics evaluated: 43


In [21]:

# Train and summarize the 75% student sample

indices_75 = saturation_samples[0.75]

docs_75 = [docs[i] for i in indices_75]
embeddings_75 = embeddings[indices_75]

umap_75 = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_75 = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

topic_model_75 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_75,
    hdbscan_model=hdbscan_75,
    verbose=True
)

topics_75, probs_75 = topic_model_75.fit_transform(
    docs_75,
    embeddings=embeddings_75
)

info_75 = topic_model_75.get_topic_info()
assigned_75 = info_75[info_75["Topic"] != -1]

n_topics_75 = len(assigned_75)
n_outliers_75 = topics_75.count(-1)
top_5_count_75 = assigned_75.nlargest(5, "Count")["Count"].sum()
assigned_count_75 = len(docs_75) - n_outliers_75

coherence_75, diversity_75, evaluated_75 = evaluate_bertopic(
    topic_model_75
)

print("75% Student Sample Summary")
print("Documents:", len(docs_75))
print("Discovered topics:", n_topics_75)
print(f"Outlier rate: {100*n_outliers_75/len(docs_75):.2f}%")
print(f"Largest topic share: {100*assigned_75['Count'].max()/len(docs_75):.2f}%")
print(f"Top 5 share of assigned documents: {100*top_5_count_75/assigned_count_75:.2f}%")
print(f"UMass coherence: {coherence_75:.4f}")
print(f"Topic diversity: {diversity_75:.4f}")
print("Topics evaluated:", evaluated_75)

display(info_75[["Topic", "Count", "Name"]].head(12))


2026-11-12 19:44:03,162 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-12 19:44:18,578 - BERTopic - Dimensionality - Completed ✓
2026-11-12 19:44:18,579 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-12 19:44:19,044 - BERTopic - Cluster - Completed ✓
2026-11-12 19:44:19,053 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-12 19:44:19,920 - BERTopic - Representation - Completed ✓


75% Student Sample Summary
Documents: 12151
Discovered topics: 83
Outlier rate: 24.06%
Largest topic share: 3.99%
Top 5 share of assigned documents: 14.10%
UMass coherence: -2.3319
Topic diversity: 0.6739
Topics evaluated: 83


,Topic,Count,Name
0,-1,2923,-1_td_th_the_torch
1,0,485,0_tensor_shape_01_print
2,1,233,1_false_true_no_4999999999999999
3,2,204,2_missing_rows_drop_how
4,3,193,3_mid_frac_probability_your
5,4,186,4_blood_feature_yes_cell
6,5,185,5_05_split_features_test
7,6,184,6_paragraph_sentences_answer_give
8,7,183,7_git_commit_github_remote
9,8,179,8_node_word_char_children


In [22]:

#Display 75% saturation evaluation results

print("75% Student Sample Summary")
print("Documents:", len(docs_75))
print("Discovered topics:", n_topics_75)
print(f"Outlier rate: {100*n_outliers_75/len(docs_75):.2f}%")
print(f"Largest topic share: {100*assigned_75['Count'].max()/len(docs_75):.2f}%")
print(f"Top 5 share of assigned documents: {100*top_5_count_75/assigned_count_75:.2f}%")
print(f"UMass coherence: {coherence_75:.4f}")
print(f"Topic diversity: {diversity_75:.4f}")
print("Topics evaluated:", evaluated_75)


75% Student Sample Summary
Documents: 12151
Discovered topics: 83
Outlier rate: 24.06%
Largest topic share: 3.99%
Top 5 share of assigned documents: 14.10%
UMass coherence: -2.3319
Topic diversity: 0.6739
Topics evaluated: 83


In [23]:

#Complete the topic saturation comparison

import pandas as pd

# Evaluate the 25% sample using the same reference corpus
coherence_25, diversity_25, evaluated_25 = evaluate_bertopic(
    topic_model_25
)

# Collect all models
models = [
    ("25%", 51, docs_25, topics_25, topic_model_25),
    ("50%", 102, docs_50pct, topics_50pct, topic_model_50pct),
    ("75%", 153, docs_75, topics_75, topic_model_75),
    ("100%", 203, docs, topics_50, topic_model_50)
]

evaluations = {
    "25%": (coherence_25, diversity_25),
    "50%": (coherence_50pct, diversity_50pct),
    "75%": (coherence_75, diversity_75),
    "100%": (coherence_50, diversity_50)
}

results = []

for label, students, sample_docs, labels, model in models:
    info = model.get_topic_info()
    assigned = info[info["Topic"] != -1]

    outliers = labels.count(-1)
    assigned_count = len(sample_docs) - outliers
    top_five = assigned.nlargest(5, "Count")["Count"].sum()

    coherence, diversity = evaluations[label]

    results.append({
        "Sample": label,
        "Students": students,
        "Documents": len(sample_docs),
        "Topics": len(assigned),
        "Outlier %": 100 * outliers / len(sample_docs),
        "Largest Topic %": (
            100 * assigned["Count"].max() / len(sample_docs)
        ),
        "Top 5 Share %": 100 * top_five / assigned_count,
        "UMass Coherence": coherence,
        "Topic Diversity": diversity
    })

saturation_results_df = pd.DataFrame(results)

display(saturation_results_df.round(4))

# Save results for our research report
saturation_results_df.to_csv(
    "bertopic_saturation_results.csv",
    index=False
)

print("Saturation comparison saved!")


,Sample,Students,Documents,Topics,Outlier %,Largest Topic %,Top 5 Share %,UMass Coherence,Topic Diversity
0,25%,51,3605,17,8.5437,19.1678,63.9976,-1.9814,0.8595
1,50%,102,7454,43,20.7942,6.3188,30.7249,-1.8469,0.7281
2,75%,153,12151,83,24.0556,3.9914,14.0984,-2.3319,0.6739
3,100%,203,16597,119,22.0341,1.5726,9.4668,-2.7437,0.6555


Saturation comparison saved!


In [24]:

# Step 33: Compare BERTopic topics across sample sizes

import numpy as np
import pandas as pd

models_by_fraction = {
    "25%": topic_model_25,
    "50%": topic_model_50pct,
    "75%": topic_model_75,
    "100%": topic_model_50
}

def get_topic_keywords(model, top_n=10):
    return {
        topic_id: set(
            word for word, _ in model.get_topic(topic_id)[:top_n]
        )
        for topic_id in model.get_topics()
        if topic_id != -1
    }

def jaccard_similarity(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0

results = []

comparisons = [
    ("25%", "50%"),
    ("50%", "75%"),
    ("75%", "100%"),
    ("25%", "100%")
]

for smaller, larger in comparisons:
    smaller_topics = get_topic_keywords(models_by_fraction[smaller])
    larger_topics = get_topic_keywords(models_by_fraction[larger])

    best_scores = []

    for words in smaller_topics.values():
        best_match = max(
            jaccard_similarity(words, other_words)
            for other_words in larger_topics.values()
        )
        best_scores.append(best_match)

    results.append({
        "Comparison": f"{smaller} → {larger}",
        "Smaller Model Topics": len(smaller_topics),
        "Larger Model Topics": len(larger_topics),
        "Mean Best Jaccard": np.mean(best_scores),
        "Median Best Jaccard": np.median(best_scores),
        "Topics with Jaccard ≥ 0.5 (%)":
            100 * np.mean(np.array(best_scores) >= 0.5)
    })

stability_df = pd.DataFrame(results)
display(stability_df.round(4))


,Comparison,Smaller Model Topics,Larger Model Topics,Mean Best Jaccard,Median Best Jaccard,Topics with Jaccard ≥ 0.5 (%)
0,25% → 50%,17,43,0.5447,0.5385,58.8235
1,50% → 75%,43,83,0.5992,0.5385,74.4186
2,75% → 100%,83,119,0.6431,0.6667,71.0843
3,25% → 100%,17,119,0.4301,0.4286,35.2941


In this step, we compare the topics found in the 75% and 100% student samples to see whether BERTopic identifies similar themes as more data is added.

Our earlier results showed that topic similarity generally increased between consecutive samples. However, this does not automatically mean the topics are stable, since larger datasets contain more topics that could potentially match.

To investigate further, we compare the 15 most similar topic pairs using their keywords and Jaccard similarity scores. We also check whether multiple topics from the 75% sample match the same topic in the full dataset.

This helps us determine whether BERTopic is consistently identifying similar themes across different sample sizes and whether the topics are becoming more stable.

In [25]:

#Inspect keyword matches between 75% and 100% models

small_topics = get_topic_keywords(topic_model_75)
full_topics = get_topic_keywords(topic_model_50)

matches = []

for small_id, small_words in small_topics.items():

    full_id, best_score = max(
        (
            (full_id, jaccard_similarity(small_words, full_words))
            for full_id, full_words in full_topics.items()
        ),
        key=lambda item: item[1]
    )

    matches.append({
        "75% Topic": small_id,
        "100% Topic": full_id,
        "Jaccard": round(best_score, 4),
        "75% Keywords": ", ".join(
            word for word, _ in topic_model_75.get_topic(small_id)[:7]
        ),
        "100% Keywords": ", ".join(
            word for word, _ in topic_model_50.get_topic(full_id)[:7]
        )
    })

matched_topics_df = (
    pd.DataFrame(matches)
    .sort_values("Jaccard", ascending=False)
)

print("Top 15 most similar topic matches:")
display(matched_topics_df.head(15))

print(
    "\nDistinct full-model topics matched:",
    matched_topics_df["100% Topic"].nunique()
)
print(
    "Total 75% topics:",
    len(matched_topics_df)
)


Top 15 most similar topic matches:


,75% Topic,100% Topic,Jaccard,75% Keywords,100% Keywords
2,2,22,1.0000,"missing, rows, drop, how, remove, nan, values","drop, remove, nan, rows, how, missing, column"
11,11,14,1.0000,"columns, rbc, categorical_columns, rbc_abnorma...","columns, rbc, categorical_columns, rbc_abnorma..."
15,15,38,1.0000,"node, children, prefix, char, self, word, sugg...","node, prefix, children, char, self, word, sugg..."
36,36,21,1.0000,"updated, redacted, main, gram, users, sequence...","updated, redacted, main, gram, users, enter, a..."
42,42,33,1.0000,"vocab, char_to_idx, text, mappings, create, da...","vocab, char_to_idx, text, mappings, create, ma..."
49,49,36,1.0000,"tuning, different, hyper, provide, architectur...","tuning, different, hyper, provide, architectur..."
44,44,27,1.0000,"loss, num_epochs, train_losses, train_loader, ...","plt, loss, num_epochs, train_losses, train_loa..."
32,32,39,1.0000,"svm, score, support, classifier, vector, svc, ...","svm, support, vector, classifier, score, svc, ..."
23,23,18,1.0000,"sort, sorting, reset, indices, according, sure...","sort, reset, sorting, indices, according, sure..."
58,58,59,1.0000,"next, character, vocabulary, possible, sequenc...","next, vocabulary, character, possible, sequenc..."



Distinct full-model topics matched: 77
Total 75% topics: 83


In [26]:
#Compare document assignments in 75% vs. 100% models

import numpy as np
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

# Labels for the same documents in the 75% sample
labels_75 = np.asarray(topics_75)

# Retrieve full-model labels for those exact documents
labels_full_on_75 = np.asarray(topics_50)[
    saturation_samples[0.75]
]

# Only compare documents assigned to topics in both models
valid = (labels_75 != -1) & (labels_full_on_75 != -1)

ari = adjusted_rand_score(
    labels_75[valid],
    labels_full_on_75[valid]
)

nmi = normalized_mutual_info_score(
    labels_75[valid],
    labels_full_on_75[valid]
)

print("75% vs. 100% Document Assignment Stability")
print("Shared documents:", len(labels_75))
print("Assigned in both models:", valid.sum())
print(f"Shared assignment coverage: {100*valid.mean():.2f}%")
print(f"Adjusted Rand Index (ARI): {ari:.4f}")
print(f"Normalized Mutual Information (NMI): {nmi:.4f}")


75% vs. 100% Document Assignment Stability
Shared documents: 12151
Assigned in both models: 7985
Shared assignment coverage: 65.71%
Adjusted Rand Index (ARI): 0.7651
Normalized Mutual Information (NMI): 0.9294


In [27]:

# Compare stability with and without outliers

import pandas as pd
import numpy as np
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

labels_75 = np.asarray(topics_75)
labels_full = np.asarray(topics_50)[saturation_samples[0.75]]

valid = (labels_75 != -1) & (labels_full != -1)

results = pd.DataFrame([
    {
        "Comparison": "Non-outliers in both models",
        "Documents": int(valid.sum()),
        "Coverage (%)": 100 * valid.mean(),
        "ARI": adjusted_rand_score(
            labels_75[valid], labels_full[valid]
        ),
        "NMI": normalized_mutual_info_score(
            labels_75[valid], labels_full[valid]
        )
    },
    {
        "Comparison": "All shared documents",
        "Documents": len(labels_75),
        "Coverage (%)": 100.0,
        "ARI": adjusted_rand_score(labels_75, labels_full),
        "NMI": normalized_mutual_info_score(labels_75, labels_full)
    }
])

display(results.round(4))

# Examine whether outlier status agrees
outlier_agreement = np.mean(
    (labels_75 == -1) == (labels_full == -1)
)

print(f"Outlier-status agreement: {outlier_agreement * 100:.2f}%")


,Comparison,Documents,Coverage (%),ARI,NMI
0,Non-outliers in both models,7985,65.7148,0.7651,0.9294
1,All shared documents,12151,100.0000,0.2688,0.7488


Outlier-status agreement: 77.51%


### BERTopic Stability and Saturation Results

BERTopic successfully identified several recurring themes in students' interactions with ChatGPT, including programming, machine learning, and data analysis. As more students were included, the number of discovered topics increased from 17 to 119, suggesting that topic saturation had not yet been reached.

Many themes remained recognizable across different sample sizes. The model also showed relatively strong consistency when comparing prompts that were assigned to topics in both models. However, consistency decreased when outliers were included, indicating that some prompts were harder to categorize reliably.

Overall, BERTopic was useful for discovering common themes in student-AI interactions, but its results were not completely stable. These findings suggest that topic modeling should be supported by additional analysis to better understand how students use generative AI for learning.